In [ ]:
from google.cloud import bigquery
import pandas as pd
import json
from datetime import datetime

In [ ]:
client = bigquery.Client(project="flb-rtl-dtl-bi")

## Definición de la Campaña

In [ ]:
campaing_name     = "Campaña Mayo 2026"
fecha_inicio_str  = "2026-05-19"
fecha_fin_str     = "2026-05-31"

# Filtro por SKU — completar con los estilos adjuntos
filter_column = "ID_ESTILO"
filter_values = []  # <- pegar aquí los SKUs, ej: [883750108, 883750118, ...]

In [ ]:
def sql_literal(v):
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return str(v)
    return "'" + str(v).replace("'", "''") + "'"

def build_in_filter(column, values):
    if not values:
        return "1=1"
    return f"{column} IN ({', '.join(sql_literal(v) for v in values)})"

dynamic_filter = build_in_filter(filter_column, filter_values)
print(dynamic_filter)

## Query — Compradores de la Campaña

In [ ]:
sql = f"""
DECLARE fecha_inicio DATE DEFAULT '{fecha_inicio_str}';
DECLARE fecha_fin    DATE DEFAULT '{fecha_fin_str}';

DECLARE max_partition DATE;
SET max_partition = (
  SELECT MAX(partition_date)
  FROM `falabella-rtl-segmentation-ai.sia_10_mkt_processes.customer_lifecycle`
  WHERE partition_date < fecha_fin
);

WITH

-- Sociodemo (género + edad)
sociodemo AS (
  SELECT CUST_ID, cust_gender, CUST_AGE AS edad
  FROM `falabella-rtl-segmentation-ai.sia_10_mkt_processes.customer_lifecycle`
  WHERE DATE(PARTITION_DATE) = max_partition
  QUALIFY ROW_NUMBER() OVER (PARTITION BY CUST_ID ORDER BY PARTITION_DATE DESC) = 1
),

-- Transacciones de campaña
trx AS (
  SELECT
    CUST_RULE_ID,
    CONCAT(CAST(CUST_RULE_ID AS STRING), '-', CAST(TRAN_DT AS STRING), '-', STORE_NAME) AS id_visita,
    PROD_SUB_CAT_ID    AS sublinea_id,
    PROD_SUB_CAT_DESC  AS sublinea_desc,
    PROD_SUB_CAT2_ID   AS clase_id,
    PROD_SUB_CAT2_DESC AS clase_desc,
    SUM(TRAN_TOTAL_AMT) AS venta,
    SUM(TRAN_QTY)       AS unidades
  FROM `falabella-rtl-segmentation-ai.sia_02_entities.cl_entities_transaction_enriched_unified`
  WHERE TRAN_DT BETWEEN fecha_inicio AND fecha_fin
    AND PARTITION_DATE BETWEEN fecha_inicio AND fecha_fin
    AND VALID_PRODUCT_FLG = TRUE
    AND (STORE_F_FLG = TRUE OR STORE_STAND_ALONE_FLG = TRUE)
    AND INVALID_CUSTOMER_FLG = FALSE
    AND TRAN_TYPE_DESC IN ('VENTA', 'FACTURA')
    AND BU_MODEL IN ('3P GSC', '3P IMP', '1P FALABELLA', '3P SVL')
    AND ({dynamic_filter})
  GROUP BY ALL
),

-- Historial previo: Falabella completa (365 días antes)
hist_fal AS (
  SELECT DISTINCT CUST_RULE_ID
  FROM `falabella-rtl-segmentation-ai.sia_02_entities.cl_entities_transaction_enriched_unified`
  WHERE TRAN_DT BETWEEN DATE_SUB(fecha_inicio, INTERVAL 365 DAY) AND DATE_SUB(fecha_inicio, INTERVAL 1 DAY)
    AND PARTITION_DATE BETWEEN DATE_SUB(fecha_inicio, INTERVAL 365 DAY) AND DATE_SUB(fecha_inicio, INTERVAL 1 DAY)
    AND VALID_PRODUCT_FLG = TRUE
    AND STORE_F_FLG = TRUE
    AND INVALID_CUSTOMER_FLG = FALSE
    AND TRAN_TYPE_DESC IN ('VENTA', 'FACTURA', 'CAMBIO', 'DEVOLUCION')
    AND BU_MODEL IN ('3P GSC', '3P IMP', '1P FALABELLA', '3P SVL')
),

-- Historial previo: Clientes Falabella (gasto > 620K Y visitas > 6)
hist_clte_f AS (
  SELECT CUST_RULE_ID
  FROM (
    SELECT
      CUST_RULE_ID,
      SUM(TRAN_TOTAL_AMT) AS venta_hist,
      COUNT(DISTINCT CONCAT(CAST(CUST_RULE_ID AS STRING), CAST(TRAN_DT AS STRING), STORE_NAME)) AS visitas_hist
    FROM `falabella-rtl-segmentation-ai.sia_02_entities.cl_entities_transaction_enriched_unified`
    WHERE TRAN_DT BETWEEN DATE_SUB(fecha_inicio, INTERVAL 365 DAY) AND DATE_SUB(fecha_inicio, INTERVAL 1 DAY)
      AND PARTITION_DATE BETWEEN DATE_SUB(fecha_inicio, INTERVAL 365 DAY) AND DATE_SUB(fecha_inicio, INTERVAL 1 DAY)
      AND VALID_PRODUCT_FLG = TRUE
      AND STORE_F_FLG = TRUE
      AND INVALID_CUSTOMER_FLG = FALSE
      AND TRAN_TYPE_DESC IN ('VENTA', 'FACTURA', 'CAMBIO', 'DEVOLUCION')
      AND BU_MODEL IN ('3P GSC', '3P IMP', '1P FALABELLA', '3P SVL')
    GROUP BY ALL
  )
  WHERE venta_hist > 620000 AND visitas_hist > 6
),

-- Historial previo: sublínea J12 completa
hist_j12 AS (
  SELECT DISTINCT CUST_RULE_ID
  FROM `falabella-rtl-segmentation-ai.sia_02_entities.cl_entities_transaction_enriched_unified`
  WHERE TRAN_DT BETWEEN DATE_SUB(fecha_inicio, INTERVAL 365 DAY) AND DATE_SUB(fecha_inicio, INTERVAL 1 DAY)
    AND PARTITION_DATE BETWEEN DATE_SUB(fecha_inicio, INTERVAL 365 DAY) AND DATE_SUB(fecha_inicio, INTERVAL 1 DAY)
    AND VALID_PRODUCT_FLG = TRUE
    AND (STORE_F_FLG = TRUE OR STORE_STAND_ALONE_FLG = TRUE)
    AND INVALID_CUSTOMER_FLG = FALSE
    AND TRAN_TYPE_DESC IN ('VENTA', 'FACTURA', 'CAMBIO', 'DEVOLUCION')
    AND BU_MODEL IN ('3P GSC', '3P IMP', '1P FALABELLA', '3P SVL')
    AND PROD_SUB_CAT_ID = 'J12'
),

-- Historial previo: clase J1201
hist_j1201 AS (
  SELECT DISTINCT CUST_RULE_ID
  FROM `falabella-rtl-segmentation-ai.sia_02_entities.cl_entities_transaction_enriched_unified`
  WHERE TRAN_DT BETWEEN DATE_SUB(fecha_inicio, INTERVAL 365 DAY) AND DATE_SUB(fecha_inicio, INTERVAL 1 DAY)
    AND PARTITION_DATE BETWEEN DATE_SUB(fecha_inicio, INTERVAL 365 DAY) AND DATE_SUB(fecha_inicio, INTERVAL 1 DAY)
    AND VALID_PRODUCT_FLG = TRUE
    AND (STORE_F_FLG = TRUE OR STORE_STAND_ALONE_FLG = TRUE)
    AND INVALID_CUSTOMER_FLG = FALSE
    AND TRAN_TYPE_DESC IN ('VENTA', 'FACTURA', 'CAMBIO', 'DEVOLUCION')
    AND BU_MODEL IN ('3P GSC', '3P IMP', '1P FALABELLA', '3P SVL')
    AND PROD_SUB_CAT2_ID = 'J1201'
),

-- Historial previo: clase J1204
hist_j1204 AS (
  SELECT DISTINCT CUST_RULE_ID
  FROM `falabella-rtl-segmentation-ai.sia_02_entities.cl_entities_transaction_enriched_unified`
  WHERE TRAN_DT BETWEEN DATE_SUB(fecha_inicio, INTERVAL 365 DAY) AND DATE_SUB(fecha_inicio, INTERVAL 1 DAY)
    AND PARTITION_DATE BETWEEN DATE_SUB(fecha_inicio, INTERVAL 365 DAY) AND DATE_SUB(fecha_inicio, INTERVAL 1 DAY)
    AND VALID_PRODUCT_FLG = TRUE
    AND (STORE_F_FLG = TRUE OR STORE_STAND_ALONE_FLG = TRUE)
    AND INVALID_CUSTOMER_FLG = FALSE
    AND TRAN_TYPE_DESC IN ('VENTA', 'FACTURA', 'CAMBIO', 'DEVOLUCION')
    AND BU_MODEL IN ('3P GSC', '3P IMP', '1P FALABELLA', '3P SVL')
    AND PROD_SUB_CAT2_ID = 'J1204'
),

base AS (
  SELECT
    t.CUST_RULE_ID,
    t.id_visita,
    t.sublinea_id,
    t.sublinea_desc,
    t.clase_id,
    t.clase_desc,
    t.venta,
    t.unidades,
    CASE WHEN hf.CUST_RULE_ID     IS NULL THEN 1 ELSE 0 END AS nuevo_fal,
    CASE WHEN hcf.CUST_RULE_ID    IS NOT NULL THEN 1 ELSE 0 END AS clte_f,
    CASE WHEN hj12.CUST_RULE_ID   IS NULL THEN 1 ELSE 0 END AS nuevo_j12,
    CASE WHEN hj1201.CUST_RULE_ID IS NULL THEN 1 ELSE 0 END AS nuevo_j1201,
    CASE WHEN hj1204.CUST_RULE_ID IS NULL THEN 1 ELSE 0 END AS nuevo_j1204,
    sd.cust_gender,
    sd.edad
  FROM trx t
  LEFT JOIN sociodemo   sd      ON t.CUST_RULE_ID = sd.CUST_ID
  LEFT JOIN hist_fal    hf      ON t.CUST_RULE_ID = hf.CUST_RULE_ID
  LEFT JOIN hist_clte_f hcf     ON t.CUST_RULE_ID = hcf.CUST_RULE_ID
  LEFT JOIN hist_j12    hj12    ON t.CUST_RULE_ID = hj12.CUST_RULE_ID
  LEFT JOIN hist_j1201  hj1201  ON t.CUST_RULE_ID = hj1201.CUST_RULE_ID
  LEFT JOIN hist_j1204  hj1204  ON t.CUST_RULE_ID = hj1204.CUST_RULE_ID
)

-- Una fila por (cliente, clase) para poder agrupar en Python
SELECT
  CUST_RULE_ID,
  sublinea_id,
  sublinea_desc,
  clase_id,
  clase_desc,
  MAX(nuevo_fal)    AS nuevo_fal,
  MAX(clte_f)       AS clte_f,
  MAX(nuevo_j12)    AS nuevo_j12,
  MAX(nuevo_j1201)  AS nuevo_j1201,
  MAX(nuevo_j1204)  AS nuevo_j1204,
  MAX(cust_gender)  AS cust_gender,
  MAX(edad)         AS edad,
  SUM(venta)        AS venta,
  SUM(unidades)     AS unidades,
  COUNT(DISTINCT id_visita) AS visitas
FROM base
GROUP BY ALL
"""

In [ ]:
df = client.query(sql).to_dataframe()
print(f"Filas resultado: {len(df):,}")
df.head()

## Output JSON

In [ ]:
MESES_ES = {1:'ene',2:'feb',3:'mar',4:'abr',5:'may',6:'jun',
            7:'jul',8:'ago',9:'sep',10:'oct',11:'nov',12:'dic'}

def fmt_periodo(ini, fin):
    d0 = datetime.strptime(ini, "%Y-%m-%d")
    d1 = datetime.strptime(fin, "%Y-%m-%d")
    return f"{d0.day:02d} {MESES_ES[d0.month]} — {d1.day:02d} {MESES_ES[d1.month]} {d1.year}"

# ── Deduplicar a nivel cliente (tomar primera aparición para datos de perfil) ──
clientes = (
    df.groupby("CUST_RULE_ID", as_index=False)
    .agg(
        nuevo_fal   = ("nuevo_fal",   "max"),
        clte_f      = ("clte_f",      "max"),
        nuevo_j12   = ("nuevo_j12",   "max"),
        nuevo_j1201 = ("nuevo_j1201", "max"),
        nuevo_j1204 = ("nuevo_j1204", "max"),
        cust_gender = ("cust_gender", "first"),
        edad        = ("edad",        "first"),
        venta       = ("venta",       "sum"),
        visitas     = ("visitas",     "sum"),
    )
)

# ── Totales ──
n_clientes    = len(clientes)
venta_total   = int(clientes["venta"].sum())
visitas_total = int(clientes["visitas"].sum())
ticket_prom   = round(venta_total / visitas_total) if visitas_total else 0

# ── Clientes nuevos ──
nuevos_fal   = int((clientes["nuevo_fal"]   == 1).sum())
cltes_f      = int((clientes["clte_f"]      == 1).sum())
nuevos_j12   = int((clientes["nuevo_j12"]   == 1).sum())
nuevos_j1201 = int((clientes["nuevo_j1201"] == 1).sum())
nuevos_j1204 = int((clientes["nuevo_j1204"] == 1).sum())

# ── Género ──
gen = clientes[clientes["cust_gender"].isin(["M", "F"])]["cust_gender"].value_counts()
gen_total = gen.sum() or 1
pct_f = round(gen.get("F", 0) / gen_total * 100, 1)
pct_m = round(gen.get("M", 0) / gen_total * 100, 1)

# ── Rango etario ──
def cnt_age(lo, hi):
    return int(((clientes["edad"] >= lo) & (clientes["edad"] <= hi)).sum())

edad_con = int(((clientes["edad"] >= 18) & (clientes["edad"] <= 99)).sum()) or 1

rango_etario = [
    {"rango": "18-25", "clientes": cnt_age(18, 25), "pct": round(cnt_age(18, 25) / edad_con * 100, 1)},
    {"rango": "26-35", "clientes": cnt_age(26, 35), "pct": round(cnt_age(26, 35) / edad_con * 100, 1)},
    {"rango": "36-45", "clientes": cnt_age(36, 45), "pct": round(cnt_age(36, 45) / edad_con * 100, 1)},
    {"rango": "46-55", "clientes": cnt_age(46, 55), "pct": round(cnt_age(46, 55) / edad_con * 100, 1)},
    {"rango": "56+",   "clientes": cnt_age(56, 120), "pct": round(cnt_age(56, 120) / edad_con * 100, 1)},
]

# ── Clientes por clase (quiénes compraron qué dentro de J12) ──
por_clase = (
    df.groupby(["clase_id", "clase_desc"])["CUST_RULE_ID"]
    .nunique()
    .reset_index()
    .rename(columns={"CUST_RULE_ID": "clientes"})
    .sort_values("clientes", ascending=False)
    .assign(pct=lambda x: (x["clientes"] / n_clientes * 100).round(1))
    .to_dict("records")
)

# ── Clientes por sublínea ──
por_sublinea = (
    df.groupby(["sublinea_id", "sublinea_desc"])["CUST_RULE_ID"]
    .nunique()
    .reset_index()
    .rename(columns={"CUST_RULE_ID": "clientes"})
    .sort_values("clientes", ascending=False)
    .assign(pct=lambda x: (x["clientes"] / n_clientes * 100).round(1))
    .to_dict("records")
)

# ── Armar JSON ──
output = {
    "meta": {
        "campana": campaing_name,
        "periodo": fmt_periodo(fecha_inicio_str, fecha_fin_str)
    },
    "totales": {
        "clientes":       n_clientes,
        "venta_total":    venta_total,
        "ticket_promedio": ticket_prom,
        "clientes_f":     cltes_f
    },
    "clientes_nuevos": {
        "falabella":   {"n": nuevos_fal,   "pct": round(nuevos_fal   / n_clientes * 100, 1)},
        "j12_completa": {"n": nuevos_j12,  "pct": round(nuevos_j12   / n_clientes * 100, 1)},
        "j1201":       {"n": nuevos_j1201, "pct": round(nuevos_j1201 / n_clientes * 100, 1)},
        "j1204":       {"n": nuevos_j1204, "pct": round(nuevos_j1204 / n_clientes * 100, 1)}
    },
    "genero": {
        "mujeres_pct": pct_f,
        "hombres_pct": pct_m
    },
    "rango_etario": rango_etario,
    "clientes_por_clase":    por_clase,
    "clientes_por_sublinea": por_sublinea
}

print(json.dumps(output, ensure_ascii=False, indent=2))